# 02 · Data Preprocessing & Patient-Independent Split

Extracts heartbeat windows from MIT-BIH and INCART databases and builds a 
patient-independent train/test split to prevent data leakage.

- Bandpass filter (0.5–40 Hz) applied to raw ECG signals
- Fixed 500-sample window centered on each R-peak
- INCART resampled to match MIT-BIH (360 Hz)
- Classes: N (normal), R (RBBB), V (PVC) — A (supraventricular) excluded in final model
- Saves: `X_train_pi.npy`, `X_test_pi.npy`, `y_train_pi.npy`, `y_test_pi.npy`

In [1]:
import wfdb
import numpy as np
from scipy.signal import resample, butter, filtfilt
from collections import Counter

# ── Constants ──────────────────────────────────────────────────────────────
WINDOW   = 250          # samples each side (MIT-BIH 360Hz)
FS_MITBIH = 360
FS_INCART = 257
TARGET_LEN = 500

LABEL_MAP = {
    'N':  'N', 'L': 'L', 'R': 'R', 'e': 'N', 'j': 'N',
    'A':  'A', 'a': 'A', 'J': 'A', 'S': 'A',
    'V':  'V', 'E': 'V',
    '/':  None, 'f': None, 'Q': None, '~': None,
    '|':  None, 'x': None, '!': None, '[': None,
    ']':  None, '+': None, 's': None, 'T': None,
    '*':  None, 'D': None, '=': None, 'p': None,
    '%':  None, 'B': None, '^': None, 't': None,
    'u':  None, '?': None, 'U': None,
}
VALID_LABELS = {'N', 'V', 'A', 'R'}

MITBIH_TRAIN = [
    '100','101','102','103','104','107','108','109','111','112',
    '113','114','115','116','117','118','121','122','123','124',
    '201','202','203','205','209','212','215','220','222','223',
    '230','232','233','234'
]
MITBIH_TEST = [
    '208','210','213','214','219','221','228','231',
    '105','106','119','200','207','217'
]

def bandpass(signal, fs, lo=0.5, hi=40):
    b, a = butter(4, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, signal, axis=0)

print('Constants loaded.')


Constants loaded.


In [2]:
# ── Extract MIT-BIH beats + RR intervals ──────────────────────────────────
def extract_mitbih(rec_id):
    rec  = wfdb.rdrecord(rec_id, pn_dir='mitdb')
    ann  = wfdb.rdann(rec_id, 'atr', pn_dir='mitdb')
    sig  = bandpass(rec.p_signal, FS_MITBIH)
    # Use channels 0 (MLII) and 1 (V5)
    ch0, ch1 = 0, 1 if rec.p_signal.shape[1] > 1 else 0

    beats, labels, rr_features = [], [], []
    peaks = ann.sample
    syms  = ann.symbol

    for i, (peak, sym) in enumerate(zip(peaks, syms)):
        lbl = LABEL_MAP.get(sym)
        if lbl not in VALID_LABELS:
            continue
        lo = peak - WINDOW
        hi = peak + WINDOW
        if lo < 0 or hi > sig.shape[0]:
            continue
        beat = np.stack([sig[lo:hi, ch0], sig[lo:hi, ch1]], axis=0).astype(np.float32)
        # RR intervals in seconds
        rr_prev = (peaks[i] - peaks[i-1]) / FS_MITBIH if i > 0 else 0.8
        rr_next = (peaks[i+1] - peaks[i]) / FS_MITBIH if i < len(peaks)-1 else 0.8
        beats.append(beat)
        labels.append(lbl)
        rr_features.append([rr_prev, rr_next])
    return beats, labels, rr_features

print('MIT-BIH extractor defined.')


MIT-BIH extractor defined.


In [3]:
# ── Extract INCART beats + RR intervals ───────────────────────────────────
def extract_incart(rec_id):
    rec  = wfdb.rdrecord(rec_id, pn_dir='incartdb')
    ann  = wfdb.rdann(rec_id, 'atr', pn_dir='incartdb')
    sig  = bandpass(rec.p_signal, FS_INCART)
    # Channel 1 = lead II, channel 10 = V5
    n_ch = rec.p_signal.shape[1]
    ch0  = 1  if n_ch > 1  else 0
    ch1  = 10 if n_ch > 10 else ch0

    # Window adjusted for INCART sampling rate (same time duration as MIT-BIH)
    win  = int(WINDOW * FS_INCART / FS_MITBIH)  # ~178 samples

    beats, labels, rr_features = [], [], []
    peaks = ann.sample
    syms  = ann.symbol

    for i, (peak, sym) in enumerate(zip(peaks, syms)):
        lbl = LABEL_MAP.get(sym)
        if lbl not in VALID_LABELS:
            continue
        lo = peak - win
        hi = peak + win
        if lo < 0 or hi > sig.shape[0]:
            continue
        seg = np.stack([sig[lo:hi, ch0], sig[lo:hi, ch1]], axis=0).astype(np.float32)
        # Resample to TARGET_LEN
        beat = resample(seg, TARGET_LEN, axis=1).astype(np.float32)
        # RR intervals in seconds
        rr_prev = (peaks[i] - peaks[i-1]) / FS_INCART if i > 0 else 0.8
        rr_next = (peaks[i+1] - peaks[i]) / FS_INCART if i < len(peaks)-1 else 0.8
        beats.append(beat)
        labels.append(lbl)
        rr_features.append([rr_prev, rr_next])
    return beats, labels, rr_features

print('INCART extractor defined.')


INCART extractor defined.


In [4]:
# ── Build splits ──────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

X_train, y_train, RR_train = [], [], []
X_test,  y_test,  RR_test  = [], [], []

print('-- MIT-BIH Train ' + '-'*40)
for rec in MITBIH_TRAIN:
    beats, labels, rr = extract_mitbih(rec)
    X_train.extend(beats); y_train.extend(labels); RR_train.extend(rr)
    if beats: print(f'  {rec} -> {len(beats)} beats  {Counter(labels)}')

print('\n-- MIT-BIH Test ' + '-'*41)
for rec in MITBIH_TEST:
    beats, labels, rr = extract_mitbih(rec)
    X_test.extend(beats); y_test.extend(labels); RR_test.extend(rr)
    if beats: print(f'  {rec} -> {len(beats)} beats  {Counter(labels)}')

print('\n-- INCART Train ' + '-'*41)
for i in range(1, 61):
    rec = f'I{i:02d}'
    beats, labels, rr = extract_incart(rec)
    X_train.extend(beats); y_train.extend(labels); RR_train.extend(rr)
    if beats: print(f'  {rec} -> {len(beats)} beats  {Counter(labels)}')

print('\n-- INCART Test ' + '-'*42)
for i in range(61, 76):
    rec = f'I{i:02d}'
    beats, labels, rr = extract_incart(rec)
    X_test.extend(beats); y_test.extend(labels); RR_test.extend(rr)
    if beats: print(f'  {rec} -> {len(beats)} beats  {Counter(labels)}')


-- MIT-BIH Train ----------------------------------------
  100 -> 2271 beats  Counter({'N': 2237, 'A': 33, 'V': 1})
  101 -> 1861 beats  Counter({'N': 1858, 'A': 3})
  102 -> 103 beats  Counter({'N': 99, 'V': 4})
  103 -> 2083 beats  Counter({'N': 2081, 'A': 2})
  104 -> 165 beats  Counter({'N': 163, 'V': 2})
  107 -> 59 beats  Counter({'V': 59})
  108 -> 1759 beats  Counter({'N': 1738, 'V': 17, 'A': 4})
  109 -> 38 beats  Counter({'V': 38})
  111 -> 1 beats  Counter({'V': 1})
  112 -> 2537 beats  Counter({'N': 2535, 'A': 2})
  113 -> 1793 beats  Counter({'N': 1787, 'A': 6})
  114 -> 1874 beats  Counter({'N': 1819, 'V': 43, 'A': 12})
  115 -> 1951 beats  Counter({'N': 1951})
  116 -> 2411 beats  Counter({'N': 2301, 'V': 109, 'A': 1})
  117 -> 1533 beats  Counter({'N': 1532, 'A': 1})
  118 -> 2277 beats  Counter({'R': 2165, 'A': 96, 'V': 16})
  121 -> 1861 beats  Counter({'N': 1859, 'A': 1, 'V': 1})
  122 -> 2474 beats  Counter({'N': 2474})
  123 -> 1517 beats  Counter({'N': 1514, 'V':

In [5]:
# ── Save arrays ───────────────────────────────────────────────────────────
le = LabelEncoder()
le.fit(['A', 'N', 'R', 'V'])

X_train = np.array(X_train, dtype=np.float32)
X_test  = np.array(X_test,  dtype=np.float32)
y_train = le.transform(y_train)
y_test  = le.transform(y_test)
RR_train = np.array(RR_train, dtype=np.float32)
RR_test  = np.array(RR_test,  dtype=np.float32)

print(f'Train shape : {X_train.shape}')
print(f'Test  shape : {X_test.shape}')
print(f'RR train    : {RR_train.shape}')
print(f'RR test     : {RR_test.shape}')
print(f'Classes     : {list(le.classes_)}')

print('\n-- Train distribution --')
for cls, count in zip(le.classes_, np.bincount(y_train)):
    print(f'  {cls} -> {count:>6} ({100*count/len(y_train):.1f}%)')
print('\n-- Test distribution (unseen patients) --')
for cls, count in zip(le.classes_, np.bincount(y_test)):
    print(f'  {cls} -> {count:>6} ({100*count/len(y_test):.1f}%)')

np.save('X_train_pi.npy',  X_train)
np.save('X_test_pi.npy',   X_test)
np.save('y_train_pi.npy',  y_train)
np.save('y_test_pi.npy',   y_test)
np.save('RR_train_pi.npy', RR_train)
np.save('RR_test_pi.npy',  RR_test)
print('\nFiles saved: X_train_pi, X_test_pi, y_train_pi, y_test_pi, RR_train_pi, RR_test_pi')


Train shape : (209063, 2, 500)
Test  shape : (58956, 2, 500)
RR train    : (209063, 2)
RR test     : (58956, 2)
Classes     : [np.str_('A'), np.str_('N'), np.str_('R'), np.str_('V')]

-- Train distribution --
  A ->   4313 (2.1%)
  N -> 176865 (84.6%)
  R ->   9079 (4.3%)
  V ->  18806 (9.0%)

-- Test distribution (unseen patients) --
  A ->    425 (0.7%)
  N ->  48759 (82.7%)
  R ->   1345 (2.3%)
  V ->   8427 (14.3%)

Files saved: X_train_pi, X_test_pi, y_train_pi, y_test_pi, RR_train_pi, RR_test_pi
